In [1]:
# ---------------------------------------------------------
# Temporal Consistency Evaluation
# Step 1: Load and preprocess original clinical notes
# ---------------------------------------------------------

from pathlib import Path
import json
import pandas as pd

In [2]:
# Project paths
PROJECT_ROOT = Path.cwd().parent.parent

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
)

EVALUATION_DIR = (
    RESULTS_DIR
    / "evaluation"
)

EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLINICAL_NOTES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "clinical_notes.csv"
)

STUDY_PATIENT_IDS_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "patients"
    / "study_patient_ids.json"
)

# Future output path for the reference temporal events.
TEMPORAL_REFERENCE_PATH = (
    EVALUATION_DIR
    / "temporal_reference_events.json"
)


In [3]:
# ---------------------------------------------------------
# Load the frozen 50-patient cohort
# ---------------------------------------------------------

with open(STUDY_PATIENT_IDS_PATH, "r") as f:
    study_patient_ids = json.load(f)

study_patient_id_set = set(
    study_patient_ids
)

print(
    "Study patients:",
    len(study_patient_ids)
)

Study patients: 50


In [4]:
# ---------------------------------------------------------
# Load original clinical notes
# ---------------------------------------------------------

notes = pd.read_csv(
    CLINICAL_NOTES_PATH
)

print(
    "Original notes:",
    len(notes)
)


Original notes: 1602


In [5]:
# ---------------------------------------------------------
# Apply EXACT frozen preprocessing used by the study
# ---------------------------------------------------------

# Remove unusable #NAME? notes.
notes_clean = notes[
    notes["clean_note_text"]
    .astype(str)
    .str.strip()
    != "#NAME?"
].copy()


# Sort chronologically before removing duplicate notes.
notes_dedup = (
    notes_clean
    .sort_values(
        ["person_id", "creation_timestamp"]
    )
    .drop_duplicates(
        subset=[
            "person_id",
            "clean_note_text"
        ],
        keep="first"
    )
    .reset_index(drop=True)
)

In [7]:
# Keep only the frozen study cohort.
temporal_notes = notes_dedup[
    notes_dedup["person_id"].isin(
        study_patient_id_set
    )
].copy()


# Convert creation_timestamp using the dataset's
# day/month/year timestamp format.
temporal_notes["creation_timestamp"] = pd.to_datetime(
    temporal_notes["creation_timestamp"],
    format="%d/%m/%Y %H:%M"
)

In [8]:
# Ensure final chronological ordering.
temporal_notes = (
    temporal_notes
    .sort_values(
        [
            "person_id",
            "creation_timestamp"
        ]
    )
    .reset_index(drop=True)
)


In [9]:

# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print(
    "After #NAME? removal:",
    len(notes_clean)
)

print(
    "After deduplication:",
    len(notes_dedup)
)

print(
    "Temporal study notes:",
    len(temporal_notes)
)

print(
    "Patients:",
    temporal_notes["person_id"].nunique()
)

print(
    "Patient IDs match frozen cohort:",
    set(
        temporal_notes["person_id"].unique()
    )
    == study_patient_id_set
)

After #NAME? removal: 1595
After deduplication: 1103
Temporal study notes: 1103
Patients: 50
Patient IDs match frozen cohort: True


In [10]:
# ---------------------------------------------------------
# Step 2: Build timestamped chronological records
# ---------------------------------------------------------

patient_temporal_records = {}

for person_id, patient_notes in temporal_notes.groupby(
    "person_id",
    sort=False
):

    # Notes are already chronologically sorted.
    record_parts = []

    for _, row in patient_notes.iterrows():

        timestamp = row["creation_timestamp"].strftime(
            "%Y-%m-%d %H:%M"
        )

        note_text = str(
            row["clean_note_text"]
        ).strip()

        record_parts.append(
            f"[{timestamp}]\n{note_text}"
        )

    patient_temporal_records[person_id] = (
        "\n\n".join(record_parts)
    )

In [11]:
# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

print(
    "Patient temporal records:",
    len(patient_temporal_records)
)

print(
    "IDs match frozen cohort:",
    set(patient_temporal_records.keys())
    == study_patient_id_set
)

# Inspect one patient before any LLM extraction.
test_person_id = study_patient_ids[0]

print(
    "\nTest patient:",
    test_person_id
)

print(
    "\nRecord preview:\n"
)

print(
    patient_temporal_records[
        test_person_id
    ][:3000]
)

Patient temporal records: 50
IDs match frozen cohort: True

Test patient: 028998ee-babc-4096-9b28-001bc2f9a84e

Record preview:

[2026-01-07 08:45]
- Patient: Tomos Ellis, 15-year-old male, presented via A&E on 07/01/26 at 08:45 with severe abdominal pain over the past 3 days. - Date of birth: 2008-05-17. - NHS number: 965833270. - Triage category: 3 - Urgent. - No known allergies reported. - Current medications: None declared.- Past medical history: None documented. - Initial assessment performed by Nurse Jamie Leigh Alexander. - ED diagnosis: Constipation. - Admiting consultant: Dr. Susan Jennifer Robson. - Decision: Proceed with baseline obs and assess pain severity.
Nurse Jamie Leigh Alexander 
NMC number: 27H5222T

[2026-01-07 09:10]
Patient: Tomos Ellis, 15-year-old male, presenting with abdominal pain.

- Event date/time: 07/01/26 at 09:10.
- Performed focused abdominal assessment by Nurse Jamie Leigh Alexander.
- Findings: Mild abdo distension, tenderness in lower abdo.

- NO (

In [12]:
from openai import OpenAI

# ---------------------------------------------------------
# OpenAI API client
# ---------------------------------------------------------
# The client is used to send requests to the evaluator model.
# It reads the OPENAI_API_KEY from the environment.

client = OpenAI()

EVALUATOR_MODEL = "gpt-5.4-mini"

In [17]:
# ---------------------------------------------------------
# Step 3: Define reference temporal event extraction prompt
# ---------------------------------------------------------

TEMPORAL_REFERENCE_SYSTEM_PROMPT = """
You are extracting clinically meaningful events from a longitudinal
clinical record for temporal-consistency evaluation.

The clinical notes are already presented in chronological order and
each note is preceded by its source timestamp.

Extract only clinically meaningful events whose temporal position is
important for understanding the patient's clinical course.

Examples of appropriate events include:
- presentation or admission
- important diagnosis or diagnostic change
- clinically important investigation and result
- initiation, discontinuation, or meaningful change in treatment
- procedure or surgery
- important complication or change in clinical condition
- clinically meaningful improvement or deterioration
- discharge or other major outcome

Do NOT create separate events for:
- repeated documentation of the same clinical event
- routine observations that do not change the clinical course
- normal or minor examination findings unless clinically important
- administrative documentation
- duplicated information
- plans that were never carried out, unless the plan itself materially
  changed clinical management
- intermediate assessments, routine follow-up reviews, or administrative
  communications unless they represent a clinically meaningful change
  in diagnosis, treatment, condition, or outcome
- Prefer a smaller set of distinct major clinical events that define the
  patient's trajectory rather than a detailed timeline of every encounter.

IMPORTANT:
- Use ONLY information explicitly present in the supplied record.
- Do not infer missing events or timestamps.
- Preserve the clinical meaning of each event.
- Consolidate repeated documentation of the same event into one event.
- Use the source timestamp associated with the note documenting the event.
- If an event is explicitly described as having occurred at another
  date/time, use that explicit event date/time instead.
- Do not determine or score temporal consistency. Only extract the
  reference events.

Return ONLY valid JSON:

{
  "events": [
    {
      "event_id": 1,
      "timestamp": "YYYY-MM-DD HH:MM",
      "event": "Concise description of the clinically meaningful event"
    }
  ]
}

Events must be returned in chronological order.
""".strip()


TEMPORAL_REFERENCE_USER_PROMPT = """
Extract the clinically meaningful temporal events from the following
longitudinal clinical record.

LONGITUDINAL CLINICAL RECORD:

{patient_record}
""".strip()

In [18]:
# ---------------------------------------------------------
# Step 4: Pilot reference temporal-event extraction
# ---------------------------------------------------------

test_person_id = "028998ee-babc-4096-9b28-001bc2f9a84e"

test_patient_record = patient_temporal_records[
    test_person_id
]

response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=[
        {
            "role": "system",
            "content": TEMPORAL_REFERENCE_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TEMPORAL_REFERENCE_USER_PROMPT.format(
                patient_record=test_patient_record
            )
        }
    ]
)

# Parse the JSON returned by the evaluator.
test_temporal_reference = json.loads(
    response.output_text
)

print(
    "Extracted events:",
    len(test_temporal_reference["events"])
)

for event in test_temporal_reference["events"]:
    print(
        event["event_id"],
        "|",
        event["timestamp"],
        "|",
        event["event"]
    )

Extracted events: 8
1 | 2026-01-07 08:45 | Presented to A&E with 3 days of severe abdominal pain and was diagnosed with constipation.
2 | 2026-01-07 09:25 | IV 0.9% sodium chloride was started for mild dehydration; abdominal X-ray and blood tests were pending.
3 | 2026-01-07 10:00 | Abdominal X-ray showed significant faecal loading, and a single 10 mg oral bisacodyl dose was given; IV fluids were continued.
4 | 2026-01-07 10:30 | ED review confirmed constipation with mild dehydration; pain had improved after fluids, and the patient was referred to paediatrics for admission and started on oral macrogol.
5 | 2026-01-07 15:30 | Physiotherapy session focused on gentle mobilisation to aid bowel motility; patient tolerated exercises well.
6 | 2026-01-08 09:00 | Partial bowel movement overnight with reduced abdominal pain and decreased distension; macrogol was continued and discharge was being considered if improvement continued.
7 | 2026-01-09 08:30 | Constipation resolved after a full bowel

In [19]:
# ---------------------------------------------------------
# Step 5: Extract reference temporal events for all patients
# ---------------------------------------------------------
# Saves after every patient so the run can safely resume.

# Load existing results if this cell is rerun later.
if TEMPORAL_REFERENCE_PATH.exists():

    with open(TEMPORAL_REFERENCE_PATH, "r") as f:
        temporal_reference_events = json.load(f)

    print(
        "Existing patients loaded:",
        len(temporal_reference_events)
    )

else:
    temporal_reference_events = {}

In [20]:
# ---------------------------------------------------------
# Preserve the approved pilot result
# ---------------------------------------------------------

if test_person_id not in temporal_reference_events:

    temporal_reference_events[
        test_person_id
    ] = test_temporal_reference

    with open(
        TEMPORAL_REFERENCE_PATH,
        "w"
    ) as f:
        json.dump(
            temporal_reference_events,
            f,
            indent=2
        )

In [21]:
# ---------------------------------------------------------
# Extract reference events for remaining patients
# ---------------------------------------------------------

for i, person_id in enumerate(
    study_patient_ids,
    start=1
):

    # Skip patients already completed.
    if person_id in temporal_reference_events:
        print(
            f"[{i}/50] {person_id} - already completed"
        )
        continue

    patient_record = patient_temporal_records[
        person_id
    ]

    response = client.responses.create(
        model=EVALUATOR_MODEL,
        input=[
            {
                "role": "system",
                "content": TEMPORAL_REFERENCE_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": TEMPORAL_REFERENCE_USER_PROMPT.format(
                    patient_record=patient_record
                )
            }
        ]
    )

    result = json.loads(
        response.output_text
    )

    # -----------------------------------------------------
    # Basic validation
    # -----------------------------------------------------

    if "events" not in result:
        raise ValueError(
            f"No events returned for {person_id}"
        )

    if len(result["events"]) == 0:
        raise ValueError(
            f"Zero events returned for {person_id}"
        )

    # Make sure required fields exist.
    for event in result["events"]:

        required_fields = {
            "event_id",
            "timestamp",
            "event"
        }

        if not required_fields.issubset(
            event.keys()
        ):
            raise ValueError(
                f"Invalid event structure for {person_id}"
            )

    # -----------------------------------------------------
    # Store result
    # -----------------------------------------------------

    temporal_reference_events[
        person_id
    ] = result

    # Save immediately after every patient.
    with open(
        TEMPORAL_REFERENCE_PATH,
        "w"
    ) as f:
        json.dump(
            temporal_reference_events,
            f,
            indent=2
        )

    print(
        f"[{i}/50] {person_id} - "
        f"{len(result['events'])} events saved"
    )


print("\nTemporal reference extraction complete.")

print(
    "Patients completed:",
    len(temporal_reference_events)
)

[1/50] 028998ee-babc-4096-9b28-001bc2f9a84e - already completed
[2/50] 04df53ea-55c1-48d9-84a1-1f15c133b29b - 15 events saved
[3/50] 05192757-942f-460d-b4ff-004ec39cc5ee - 20 events saved
[4/50] 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf - 18 events saved
[5/50] 0f438665-d430-4adb-8acc-c3beed9e4942 - 11 events saved
[6/50] 136c7916-4f9b-4e5c-bf01-77e9d2c681a2 - 22 events saved
[7/50] 137b8481-4f1d-4b7f-babd-20f7117023ad - 21 events saved
[8/50] 1705dd0f-011a-492c-b006-b27e03f2f4ed - 11 events saved
[9/50] 1dbe23dc-0d1e-431b-81eb-497282b46a14 - 14 events saved
[10/50] 28570119-9cdc-4120-98c0-4edb76cf36a3 - 11 events saved
[11/50] 29ea304f-821d-474e-81a1-394ca3945e02 - 13 events saved
[12/50] 31f9612b-5a6b-48ea-887b-895772a83b99 - 12 events saved
[13/50] 359014a1-10e6-4bd8-9ba7-513d021c971e - 12 events saved
[14/50] 37b5ce4d-dcfd-4bb7-bee4-d597eb114703 - 15 events saved
[15/50] 420df33b-0072-4124-b1c6-0589daef3677 - 18 events saved
[16/50] 42149ae1-6a3c-471e-a002-cb7263e8bb8c - 20 events saved

In [22]:
# ---------------------------------------------------------
# Step 6: Validate saved reference temporal events
# ---------------------------------------------------------

event_counts = {
    person_id: len(data["events"])
    for person_id, data in temporal_reference_events.items()
}

total_events = sum(event_counts.values())

print("Patients:", len(temporal_reference_events))
print("Total reference events:", total_events)
print("Minimum events per patient:", min(event_counts.values()))
print("Maximum events per patient:", max(event_counts.values()))
print(
    "Average events per patient:",
    round(total_events / len(event_counts), 2)
)

# Check exact cohort match.
reference_patient_ids = set(
    temporal_reference_events.keys()
)

print(
    "Patient IDs match frozen cohort:",
    reference_patient_ids == study_patient_id_set
)

print(
    "Missing patients:",
    study_patient_id_set - reference_patient_ids
)

print(
    "Extra patients:",
    reference_patient_ids - study_patient_id_set
)


# ---------------------------------------------------------
# Validate event structure and timestamps
# ---------------------------------------------------------

invalid_events = []

for person_id, data in temporal_reference_events.items():

    for event in data["events"]:

        # Check required fields.
        if not all(
            field in event
            for field in ["event_id", "timestamp", "event"]
        ):
            invalid_events.append(
                (person_id, event, "missing field")
            )
            continue

        # Check timestamp can be parsed.
        try:
            pd.to_datetime(
                event["timestamp"],
                format="%Y-%m-%d %H:%M"
            )
        except Exception:
            invalid_events.append(
                (person_id, event, "invalid timestamp")
            )

print(
    "Invalid events:",
    len(invalid_events)
)

Patients: 50
Total reference events: 658
Minimum events per patient: 7
Maximum events per patient: 23
Average events per patient: 13.16
Patient IDs match frozen cohort: True
Missing patients: set()
Extra patients: set()
Invalid events: 0


In [23]:
# ---------------------------------------------------------
# Step 7: Load final summaries from all four workflows
# ---------------------------------------------------------

workflow_paths = {
    "direct": (
        RESULTS_DIR
        / "direct"
        / "direct_summaries.json"
    ),
    "hierarchical": (
        RESULTS_DIR
        / "hierarchical"
        / "hierarchical_summaries.json"
    ),
    "rag": (
        RESULTS_DIR
        / "rag"
        / "final_rag_summaries.json"
    ),
    "rag_verification": (
        RESULTS_DIR
        / "rag_verification"
        / "final_verified_summaries.json"
    ),
}


# Each workflow stores its final summary under a different field.
summary_fields = {
    "direct": "summary",
    "hierarchical": "final_summary",
    "rag": "summary",
    "rag_verification": "final_summary",
}


workflow_summaries = {}


# ---------------------------------------------------------
# Load summaries
# ---------------------------------------------------------

for workflow, path in workflow_paths.items():

    with open(path, "r") as f:
        results = json.load(f)

    summary_field = summary_fields[workflow]

    workflow_summaries[workflow] = {
        person_id: data[summary_field]
        for person_id, data in results.items()
    }


# ---------------------------------------------------------
# Validate all four workflows
# ---------------------------------------------------------

for workflow, summaries in workflow_summaries.items():

    workflow_ids = set(summaries.keys())

    print(
        workflow,
        "| summaries:",
        len(summaries),
        "| IDs match:",
        workflow_ids == study_patient_id_set
    )

direct | summaries: 50 | IDs match: True
hierarchical | summaries: 50 | IDs match: True
rag | summaries: 50 | IDs match: True
rag_verification | summaries: 50 | IDs match: True


In [24]:
# ---------------------------------------------------------
# Step 8: Define temporal consistency matching prompt
# ---------------------------------------------------------

TEMPORAL_MATCH_SYSTEM_PROMPT = """
You are evaluating temporal consistency in longitudinal clinical summaries.

You will receive:
1. A chronologically ordered reference list of clinically meaningful
   events extracted from the patient's original clinical record.
2. Four generated longitudinal clinical summaries.

For each reference event, determine whether that event is represented
in each summary.

If the event is represented, assign its position in the temporal
sequence expressed by that summary.

IMPORTANT RULES:

- Match events by clinical meaning, not exact wording.
- Reasonable paraphrases count as matches.
- Do not require the summary to state the source timestamp explicitly.
- Use dates, temporal expressions, narrative order, and other explicit
  temporal cues in the summary to determine the event's expressed order.
- Do not assume that narrative sentence order alone represents chronology
  when the summary explicitly indicates a different temporal relationship.
- If a reference event is not represented, return null.
- A missing event is NOT a temporal error. Missing information is
  evaluated separately under completeness.
- Do not evaluate factual accuracy or completeness here.
- Do not invent events or temporal relationships.
- Events occurring at the same expressed temporal position may receive
  the same position.
- Apply the same matching standard to all four workflows.

Return ONLY valid JSON:

{
  "matches": [
    {
      "event_id": 1,
      "direct": 1,
      "hierarchical": 1,
      "rag": null,
      "rag_verification": 1
    }
  ]
}

Each reference event must appear exactly once.

For each workflow:
- integer = event is represented, and the integer indicates its temporal
  position in that summary
- null = reference event is not represented
""".strip()


TEMPORAL_MATCH_USER_PROMPT = """
REFERENCE EVENTS:
{reference_events}

DIRECT SUMMARY:
{direct_summary}

HIERARCHICAL SUMMARY:
{hierarchical_summary}

RAG SUMMARY:
{rag_summary}

RAG + VERIFICATION SUMMARY:
{rag_verification_summary}
""".strip()

In [25]:
# ---------------------------------------------------------
# Step 9: Pilot temporal matching on one patient
# ---------------------------------------------------------

test_person_id = "028998ee-babc-4096-9b28-001bc2f9a84e"

# Use the frozen reference events for this patient.
test_reference_events = temporal_reference_events[
    test_person_id
]["events"]

response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=[
        {
            "role": "system",
            "content": TEMPORAL_MATCH_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TEMPORAL_MATCH_USER_PROMPT.format(
                reference_events=json.dumps(
                    test_reference_events,
                    indent=2
                ),
                direct_summary=workflow_summaries[
                    "direct"
                ][test_person_id],
                hierarchical_summary=workflow_summaries[
                    "hierarchical"
                ][test_person_id],
                rag_summary=workflow_summaries[
                    "rag"
                ][test_person_id],
                rag_verification_summary=workflow_summaries[
                    "rag_verification"
                ][test_person_id]
            )
        }
    ]
)

test_temporal_matches = json.loads(
    response.output_text
)

print(
    "Reference events:",
    len(test_reference_events)
)

print(
    "Returned matches:",
    len(test_temporal_matches["matches"])
)

for match in test_temporal_matches["matches"]:
    print(
        match["event_id"],
        "| Direct:", match["direct"],
        "| Hierarchical:", match["hierarchical"],
        "| RAG:", match["rag"],
        "| RAG+Verification:", match["rag_verification"]
    )

Reference events: 8
Returned matches: 8
1 | Direct: 1 | Hierarchical: 1 | RAG: 1 | RAG+Verification: 1
2 | Direct: 2 | Hierarchical: 2 | RAG: 2 | RAG+Verification: 2
3 | Direct: 3 | Hierarchical: 2 | RAG: 2 | RAG+Verification: 2
4 | Direct: 4 | Hierarchical: 2 | RAG: 2 | RAG+Verification: 2
5 | Direct: 5 | Hierarchical: 3 | RAG: 3 | RAG+Verification: 3
6 | Direct: 6 | Hierarchical: 4 | RAG: 4 | RAG+Verification: 4
7 | Direct: 7 | Hierarchical: 5 | RAG: 5 | RAG+Verification: 5
8 | Direct: 8 | Hierarchical: 5 | RAG: 5 | RAG+Verification: 5


In [26]:
# ---------------------------------------------------------
# Inspect pilot event matching before freezing
# ---------------------------------------------------------

match_by_id = {
    match["event_id"]: match
    for match in test_temporal_matches["matches"]
}

for event in test_reference_events:

    match = match_by_id[event["event_id"]]

    print("\nREFERENCE EVENT", event["event_id"])
    print(event["event"])

    print(
        "Positions:",
        "Direct =", match["direct"],
        "| Hierarchical =", match["hierarchical"],
        "| RAG =", match["rag"],
        "| RAG+Verification =", match["rag_verification"]
    )


REFERENCE EVENT 1
Presented to A&E with 3 days of severe abdominal pain and was diagnosed with constipation.
Positions: Direct = 1 | Hierarchical = 1 | RAG = 1 | RAG+Verification = 1

REFERENCE EVENT 2
IV 0.9% sodium chloride was started for mild dehydration; abdominal X-ray and blood tests were pending.
Positions: Direct = 2 | Hierarchical = 2 | RAG = 2 | RAG+Verification = 2

REFERENCE EVENT 3
Abdominal X-ray showed significant faecal loading, and a single 10 mg oral bisacodyl dose was given; IV fluids were continued.
Positions: Direct = 3 | Hierarchical = 2 | RAG = 2 | RAG+Verification = 2

REFERENCE EVENT 4
ED review confirmed constipation with mild dehydration; pain had improved after fluids, and the patient was referred to paediatrics for admission and started on oral macrogol.
Positions: Direct = 4 | Hierarchical = 2 | RAG = 2 | RAG+Verification = 2

REFERENCE EVENT 5
Physiotherapy session focused on gentle mobilisation to aid bowel motility; patient tolerated exercises well.
P

In [27]:
# ---------------------------------------------------------
# Step 10: Run temporal matching for all 50 patients
# ---------------------------------------------------------
# Saves after every patient so the run can safely resume.

TEMPORAL_MATCH_RESULTS_PATH = (
    EVALUATION_DIR
    / "temporal_match_results.json"
)


# Load existing results if this cell is rerun later.
if TEMPORAL_MATCH_RESULTS_PATH.exists():

    with open(TEMPORAL_MATCH_RESULTS_PATH, "r") as f:
        temporal_match_results = json.load(f)

    print(
        "Existing patients loaded:",
        len(temporal_match_results)
    )

else:
    temporal_match_results = {}


# ---------------------------------------------------------
# Preserve the approved pilot result
# ---------------------------------------------------------

if test_person_id not in temporal_match_results:

    temporal_match_results[
        test_person_id
    ] = test_temporal_matches

    with open(
        TEMPORAL_MATCH_RESULTS_PATH,
        "w"
    ) as f:
        json.dump(
            temporal_match_results,
            f,
            indent=2
        )


# ---------------------------------------------------------
# Evaluate remaining patients
# ---------------------------------------------------------

for i, person_id in enumerate(
    study_patient_ids,
    start=1
):

    # Skip patients already evaluated.
    if person_id in temporal_match_results:
        print(
            f"[{i}/50] {person_id} - already completed"
        )
        continue

    reference_events = temporal_reference_events[
        person_id
    ]["events"]

    response = client.responses.create(
        model=EVALUATOR_MODEL,
        input=[
            {
                "role": "system",
                "content": TEMPORAL_MATCH_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": TEMPORAL_MATCH_USER_PROMPT.format(
                    reference_events=json.dumps(
                        reference_events,
                        indent=2
                    ),
                    direct_summary=workflow_summaries[
                        "direct"
                    ][person_id],
                    hierarchical_summary=workflow_summaries[
                        "hierarchical"
                    ][person_id],
                    rag_summary=workflow_summaries[
                        "rag"
                    ][person_id],
                    rag_verification_summary=workflow_summaries[
                        "rag_verification"
                    ][person_id]
                )
            }
        ]
    )

    result = json.loads(
        response.output_text
    )

    # -----------------------------------------------------
    # Validate returned matches
    # -----------------------------------------------------

    if "matches" not in result:
        raise ValueError(
            f"No matches returned for {person_id}"
        )

    expected_ids = {
        event["event_id"]
        for event in reference_events
    }

    returned_ids = {
        match["event_id"]
        for match in result["matches"]
    }

    # Every reference event must be evaluated exactly once.
    if len(result["matches"]) != len(reference_events):
        raise ValueError(
            f"Wrong match count for {person_id}"
        )

    if returned_ids != expected_ids:
        raise ValueError(
            f"Event ID mismatch for {person_id}"
        )

    # Validate each workflow value:
    # it must be either null (None) or a positive integer position.
    for match in result["matches"]:

        for workflow in [
            "direct",
            "hierarchical",
            "rag",
            "rag_verification"
        ]:

            value = match[workflow]

            if value is not None:

                if (
                    not isinstance(value, int)
                    or value < 1
                ):
                    raise ValueError(
                        f"Invalid position for "
                        f"{person_id}, "
                        f"event {match['event_id']}, "
                        f"{workflow}: {value}"
                    )

    # -----------------------------------------------------
    # Store and immediately save
    # -----------------------------------------------------

    temporal_match_results[
        person_id
    ] = result

    with open(
        TEMPORAL_MATCH_RESULTS_PATH,
        "w"
    ) as f:
        json.dump(
            temporal_match_results,
            f,
            indent=2
        )

    print(
        f"[{i}/50] {person_id} - "
        f"{len(result['matches'])} events evaluated"
    )


print("\nTemporal matching complete.")

print(
    "Patients completed:",
    len(temporal_match_results)
)

[1/50] 028998ee-babc-4096-9b28-001bc2f9a84e - already completed
[2/50] 04df53ea-55c1-48d9-84a1-1f15c133b29b - 15 events evaluated
[3/50] 05192757-942f-460d-b4ff-004ec39cc5ee - 20 events evaluated
[4/50] 0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf - 18 events evaluated
[5/50] 0f438665-d430-4adb-8acc-c3beed9e4942 - 11 events evaluated
[6/50] 136c7916-4f9b-4e5c-bf01-77e9d2c681a2 - 22 events evaluated
[7/50] 137b8481-4f1d-4b7f-babd-20f7117023ad - 21 events evaluated
[8/50] 1705dd0f-011a-492c-b006-b27e03f2f4ed - 11 events evaluated
[9/50] 1dbe23dc-0d1e-431b-81eb-497282b46a14 - 14 events evaluated
[10/50] 28570119-9cdc-4120-98c0-4edb76cf36a3 - 11 events evaluated
[11/50] 29ea304f-821d-474e-81a1-394ca3945e02 - 13 events evaluated
[12/50] 31f9612b-5a6b-48ea-887b-895772a83b99 - 12 events evaluated
[13/50] 359014a1-10e6-4bd8-9ba7-513d021c971e - 12 events evaluated
[14/50] 37b5ce4d-dcfd-4bb7-bee4-d597eb114703 - 15 events evaluated
[15/50] 420df33b-0072-4124-b1c6-0589daef3677 - 18 events evaluated
[16/50

In [28]:
# ---------------------------------------------------------
# Step 11: Calculate temporal consistency scores
# ---------------------------------------------------------

from itertools import combinations

workflows = [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification"
]

temporal_score_records = []


for person_id in study_patient_ids:

    # -----------------------------------------------------
    # Build reference timestamp lookup
    # -----------------------------------------------------

    reference_events = temporal_reference_events[
        person_id
    ]["events"]

    reference_timestamps = {
        event["event_id"]: pd.to_datetime(
            event["timestamp"],
            format="%Y-%m-%d %H:%M"
        )
        for event in reference_events
    }

    # LLM-produced event matches/positions.
    matches = temporal_match_results[
        person_id
    ]["matches"]

    match_lookup = {
        match["event_id"]: match
        for match in matches
    }


    # -----------------------------------------------------
    # Score each workflow independently
    # -----------------------------------------------------

    for workflow in workflows:

        correct_pairs = 0
        reversed_pairs = 0
        tied_summary_pairs = 0
        same_source_time_pairs = 0
        missing_event_pairs = 0

        event_ids = list(reference_timestamps.keys())

        # Compare every possible pair of reference events.
        for event_a, event_b in combinations(
            event_ids,
            2
        ):

            source_time_a = reference_timestamps[event_a]
            source_time_b = reference_timestamps[event_b]

            summary_pos_a = match_lookup[
                event_a
            ][workflow]

            summary_pos_b = match_lookup[
                event_b
            ][workflow]


            # -------------------------------------------------
            # Missing events are completeness, not temporal errors.
            # -------------------------------------------------

            if (
                summary_pos_a is None
                or summary_pos_b is None
            ):
                missing_event_pairs += 1
                continue


            # -------------------------------------------------
            # If source events have the same timestamp, there is
            # no reference ordering to test.
            # -------------------------------------------------

            if source_time_a == source_time_b:
                same_source_time_pairs += 1
                continue


            # -------------------------------------------------
            # If the summary expresses both events at the same
            # temporal position, treat this as an unresolved tie.
            # It is excluded rather than counted as an error.
            # -------------------------------------------------

            if summary_pos_a == summary_pos_b:
                tied_summary_pairs += 1
                continue


            # -------------------------------------------------
            # Compare source order with summary order.
            # -------------------------------------------------

            source_order = (
                source_time_a < source_time_b
            )

            summary_order = (
                summary_pos_a < summary_pos_b
            )

            if source_order == summary_order:
                correct_pairs += 1
            else:
                reversed_pairs += 1


        # -----------------------------------------------------
        # Temporal consistency score
        # -----------------------------------------------------

        evaluated_pairs = (
            correct_pairs + reversed_pairs
        )

        if evaluated_pairs > 0:
            temporal_score = (
                correct_pairs
                / evaluated_pairs
                * 100
            )
        else:
            temporal_score = None


        temporal_score_records.append(
            {
                "person_id": person_id,
                "workflow": workflow,
                "correct_pairs": correct_pairs,
                "reversed_pairs": reversed_pairs,
                "evaluated_pairs": evaluated_pairs,
                "tied_summary_pairs": tied_summary_pairs,
                "same_source_time_pairs": same_source_time_pairs,
                "missing_event_pairs": missing_event_pairs,
                "temporal_score": temporal_score,
            }
        )


# ---------------------------------------------------------
# Convert to dataframe
# ---------------------------------------------------------

temporal_scores_df = pd.DataFrame(
    temporal_score_records
)


print(
    "Rows:",
    len(temporal_scores_df)
)

print(
    "\nPatients per workflow:"
)

print(
    temporal_scores_df
    .groupby("workflow")["person_id"]
    .nunique()
)

print(
    "\nMissing temporal scores:",
    temporal_scores_df[
        "temporal_score"
    ].isna().sum()
)

print(
    "\nScore range:",
    temporal_scores_df["temporal_score"].min(),
    "to",
    temporal_scores_df["temporal_score"].max()
)

Rows: 200

Patients per workflow:
workflow
direct              50
hierarchical        50
rag                 50
rag_verification    50
Name: person_id, dtype: int64

Missing temporal scores: 16

Score range: 88.0 to 100.0


In [29]:
# ---------------------------------------------------------
# Step 12: Inspect patients with no temporal score
# ---------------------------------------------------------

missing_temporal_scores = temporal_scores_df[
    temporal_scores_df["temporal_score"].isna()
].copy()

print(
    "Missing scores:",
    len(missing_temporal_scores)
)

print(
    "\nMissing scores by workflow:"
)

print(
    missing_temporal_scores[
        "workflow"
    ].value_counts()
)

print(
    "\nDetails:"
)

print(
    missing_temporal_scores[
        [
            "person_id",
            "workflow",
            "correct_pairs",
            "reversed_pairs",
            "evaluated_pairs",
            "tied_summary_pairs",
            "same_source_time_pairs",
            "missing_event_pairs",
        ]
    ].to_string(index=False)
)

Missing scores: 16

Missing scores by workflow:
workflow
direct              4
hierarchical        4
rag                 4
rag_verification    4
Name: count, dtype: int64

Details:
                           person_id         workflow  correct_pairs  reversed_pairs  evaluated_pairs  tied_summary_pairs  same_source_time_pairs  missing_event_pairs
0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf           direct              0               0                0                 153                       0                    0
0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf     hierarchical              0               0                0                 153                       0                    0
0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf              rag              0               0                0                  91                       0                   62
0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf rag_verification              0               0                0                  78                       0      

In [30]:
# ---------------------------------------------------------
# Step 13: Inspect one zero-evaluable-pair patient
# ---------------------------------------------------------

problem_person_id = (
    "0c04d9fb-d8d0-4ee0-8fc9-0c3f7f148bdf"
)

print("REFERENCE EVENTS\n")

for event in temporal_reference_events[
    problem_person_id
]["events"]:
    print(
        event["event_id"],
        "|",
        event["timestamp"],
        "|",
        event["event"]
    )


print("\n\nDIRECT SUMMARY\n")

print(
    workflow_summaries[
        "direct"
    ][problem_person_id]
)


print("\n\nTEMPORAL POSITIONS\n")

problem_matches = temporal_match_results[
    problem_person_id
]["matches"]

for match in problem_matches:
    print(
        match["event_id"],
        "| Direct:",
        match["direct"],
        "| Hierarchical:",
        match["hierarchical"],
        "| RAG:",
        match["rag"],
        "| RAG+Verification:",
        match["rag_verification"]
    )

REFERENCE EVENTS

1 | 2025-12-20 10:00 | Pre-operative assessment for elective total left hip replacement due to end-stage osteoarthritis; mild anaemia identified (Hb 10.8 g/dL), mild hypertension noted, and ferrous sulfate started.
2 | 2025-12-20 11:30 | Anaesthetic assessment for elective left total hip replacement; BP remained elevated, GA with spinal block was planned, omeprazole was started, and the patient consented to the anaesthetic plan.
3 | 2025-12-20 12:00 | Surgical consent discussion completed for elective total left hip replacement; risks, benefits, recovery, and postoperative pain management were explained and the consent form was signed.
4 | 2025-12-20 12:30 | Pre-operative checklist completed; planned hip replacement, fasting instructions, and discontinuation of ibuprofen 48 hours before surgery were confirmed.
5 | 2026-01-02 15:20 | Admission for planned total left hip replacement; patient clinically stable with no signs of infection and surgery preparations initiated

In [35]:
# ---------------------------------------------------------
# Step 16A: Temporal ordering prompt — one summary per call
# ---------------------------------------------------------

TEMPORAL_SINGLE_SYSTEM_PROMPT = """
You are evaluating temporal consistency in a longitudinal clinical summary.

You will receive:
1. A list of clinically meaningful reference events from the patient's
   original clinical record.
2. ONE generated longitudinal clinical summary.

Your task is to identify which reference events are represented in the
summary and arrange those represented events according to the chronology
CONVEYED BY THE SUMMARY.

IMPORTANT RULES:

- Match events by clinical meaning, not exact wording.
- Reasonable paraphrases count as matches.
- Determine chronology ONLY from the generated summary.
- The reference-event list is provided only for semantic matching.
- Do NOT assume the reference-event list is chronologically ordered.
- Do NOT infer chronology from event IDs.
- Do NOT use information that appears only in the reference events to
  determine the summary's chronology.

To determine chronology, use evidence in the summary such as:
- explicit dates or times
- preoperative / postoperative
- before / after
- initially / subsequently / later / then
- over the following days
- by a stated date
- admission -> treatment/procedure -> recovery -> discharge
- narrative sequence when the summary clearly describes a progressing
  clinical course

Include only reference events that are represented in the summary.

Place events in DIFFERENT ordered groups whenever the summary conveys
that one occurred before another.

Place events in the SAME group only when:
1. the summary represents them as occurring at the same temporal stage, or
2. the summary genuinely provides no basis for determining their
   relative temporal order.

Do NOT place an entire longitudinal course into one group merely because
exact dates are absent. Clinical progression and temporal language can
establish relative chronology without exact dates.

Missing events are not temporal errors and should simply be omitted.

Return ONLY valid JSON:

{
  "ordered_event_groups": [
    [1, 2],
    [4],
    [5, 6],
    [8]
  ]
}

The outer list is chronological.
IDs within the same inner list have unresolved or equivalent relative
temporal order.
Each matched event ID must appear exactly once.
""".strip()


TEMPORAL_SINGLE_USER_PROMPT = """
REFERENCE EVENTS:

{reference_events}


GENERATED SUMMARY:

{summary}
""".strip()

In [36]:
# ---------------------------------------------------------
# Step 16B: Pilot single-summary temporal ordering
# ---------------------------------------------------------

reference_events_for_matching = [
    {
        "event_id": event["event_id"],
        "event": event["event"]
    }
    for event in temporal_reference_events[
        problem_person_id
    ]["events"]
]

response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=[
        {
            "role": "system",
            "content": TEMPORAL_SINGLE_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": TEMPORAL_SINGLE_USER_PROMPT.format(
                reference_events=json.dumps(
                    reference_events_for_matching,
                    indent=2
                ),
                summary=workflow_summaries[
                    "direct"
                ][problem_person_id]
            )
        }
    ]
)

single_temporal_pilot = json.loads(
    response.output_text
)

print(
    single_temporal_pilot[
        "ordered_event_groups"
    ]
)

[[1, 2, 3, 4], [5, 6], [7], [8], [9, 10], [11], [12, 13, 14], [15, 16], [17, 18]]


In [37]:
# ---------------------------------------------------------
# Step 17: Final temporal-order evaluation
# One patient × one workflow per evaluator call
# ---------------------------------------------------------

TEMPORAL_FINAL_ORDER_PATH = (
    EVALUATION_DIR
    / "temporal_final_order_results.json"
)

workflows = [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification"
]


# ---------------------------------------------------------
# Load existing results if resuming
# ---------------------------------------------------------

if TEMPORAL_FINAL_ORDER_PATH.exists():

    with open(TEMPORAL_FINAL_ORDER_PATH, "r") as f:
        temporal_final_order_results = json.load(f)

    print(
        "Existing patients loaded:",
        len(temporal_final_order_results)
    )

else:
    temporal_final_order_results = {}


# ---------------------------------------------------------
# Evaluate each patient and workflow independently
# ---------------------------------------------------------

for patient_index, person_id in enumerate(
    study_patient_ids,
    start=1
):

    # Create patient container if needed.
    if person_id not in temporal_final_order_results:
        temporal_final_order_results[person_id] = {}

    # Reference events WITHOUT timestamps.
    reference_events_for_matching = [
        {
            "event_id": event["event_id"],
            "event": event["event"]
        }
        for event in temporal_reference_events[
            person_id
        ]["events"]
    ]

    valid_event_ids = {
        event["event_id"]
        for event in temporal_reference_events[
            person_id
        ]["events"]
    }

    for workflow in workflows:

        # Skip completed workflow if resuming.
        if workflow in temporal_final_order_results[
            person_id
        ]:
            print(
                f"[{patient_index}/50] "
                f"{workflow} - already completed"
            )
            continue

        response = client.responses.create(
            model=EVALUATOR_MODEL,
            input=[
                {
                    "role": "system",
                    "content": TEMPORAL_SINGLE_SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": TEMPORAL_SINGLE_USER_PROMPT.format(
                        reference_events=json.dumps(
                            reference_events_for_matching,
                            indent=2
                        ),
                        summary=workflow_summaries[
                            workflow
                        ][person_id]
                    )
                }
            ]
        )

        result = json.loads(
            response.output_text
        )

        # -------------------------------------------------
        # Validate result
        # -------------------------------------------------

        if "ordered_event_groups" not in result:
            raise ValueError(
                f"Missing ordered_event_groups: "
                f"{person_id}, {workflow}"
            )

        groups = result["ordered_event_groups"]

        # Flatten returned event IDs.
        returned_ids = [
            event_id
            for group in groups
            for event_id in group
        ]

        # No duplicate event IDs.
        if len(returned_ids) != len(set(returned_ids)):
            raise ValueError(
                f"Duplicate event ID: "
                f"{person_id}, {workflow}"
            )

        # All returned IDs must belong to this patient.
        if not set(returned_ids).issubset(
            valid_event_ids
        ):
            raise ValueError(
                f"Invalid event ID: "
                f"{person_id}, {workflow}"
            )

        # -------------------------------------------------
        # Store result
        # -------------------------------------------------

        temporal_final_order_results[
            person_id
        ][workflow] = result

        # Save after EVERY workflow call.
        with open(
            TEMPORAL_FINAL_ORDER_PATH,
            "w"
        ) as f:
            json.dump(
                temporal_final_order_results,
                f,
                indent=2
            )

        print(
            f"[{patient_index}/50] "
            f"{workflow} - completed"
        )


print("\nFinal temporal ordering complete.")


# ---------------------------------------------------------
# Final completion validation
# ---------------------------------------------------------

completed_calls = sum(
    len(patient_results)
    for patient_results
    in temporal_final_order_results.values()
)

print(
    "Patients:",
    len(temporal_final_order_results)
)

print(
    "Patient-workflow evaluations:",
    completed_calls
)

[1/50] direct - completed
[1/50] hierarchical - completed
[1/50] rag - completed
[1/50] rag_verification - completed
[2/50] direct - completed
[2/50] hierarchical - completed
[2/50] rag - completed
[2/50] rag_verification - completed
[3/50] direct - completed
[3/50] hierarchical - completed
[3/50] rag - completed
[3/50] rag_verification - completed
[4/50] direct - completed
[4/50] hierarchical - completed
[4/50] rag - completed
[4/50] rag_verification - completed
[5/50] direct - completed
[5/50] hierarchical - completed
[5/50] rag - completed
[5/50] rag_verification - completed
[6/50] direct - completed
[6/50] hierarchical - completed
[6/50] rag - completed
[6/50] rag_verification - completed
[7/50] direct - completed
[7/50] hierarchical - completed
[7/50] rag - completed
[7/50] rag_verification - completed
[8/50] direct - completed
[8/50] hierarchical - completed
[8/50] rag - completed
[8/50] rag_verification - completed
[9/50] direct - completed
[9/50] hierarchical - completed
[9/50]

In [38]:
# ---------------------------------------------------------
# Step 18: Calculate FINAL temporal consistency scores
# ---------------------------------------------------------

from itertools import combinations

final_temporal_score_records = []

workflows = [
    "direct",
    "hierarchical",
    "rag",
    "rag_verification"
]


for person_id in study_patient_ids:

    reference_events = temporal_reference_events[
        person_id
    ]["events"]

    # Original source timestamp for each frozen reference event.
    reference_timestamps = {
        event["event_id"]: pd.to_datetime(
            event["timestamp"],
            format="%Y-%m-%d %H:%M"
        )
        for event in reference_events
    }

    all_event_ids = list(
        reference_timestamps.keys()
    )


    for workflow in workflows:

        groups = temporal_final_order_results[
            person_id
        ][workflow]["ordered_event_groups"]

        # -------------------------------------------------
        # Convert ordered groups into:
        # event_id -> temporal group position
        #
        # Example:
        # [[1, 2], [3], [4, 5]]
        #
        # becomes:
        # 1 -> 0
        # 2 -> 0
        # 3 -> 1
        # 4 -> 2
        # 5 -> 2
        # -------------------------------------------------

        summary_positions = {}

        for group_position, group in enumerate(groups):

            for event_id in group:
                summary_positions[
                    event_id
                ] = group_position


        correct_pairs = 0
        reversed_pairs = 0
        tied_summary_pairs = 0
        same_source_time_pairs = 0
        missing_event_pairs = 0


        # -------------------------------------------------
        # Compare every possible reference-event pair
        # -------------------------------------------------

        for event_a, event_b in combinations(
            all_event_ids,
            2
        ):

            # Missing event = completeness issue,
            # not temporal failure.
            if (
                event_a not in summary_positions
                or event_b not in summary_positions
            ):
                missing_event_pairs += 1
                continue

            source_time_a = reference_timestamps[
                event_a
            ]

            source_time_b = reference_timestamps[
                event_b
            ]

            # Same source timestamp means there is no
            # reference order to test.
            if source_time_a == source_time_b:
                same_source_time_pairs += 1
                continue

            summary_pos_a = summary_positions[
                event_a
            ]

            summary_pos_b = summary_positions[
                event_b
            ]

            # Same summary group = chronology between
            # these events is unresolved.
            if summary_pos_a == summary_pos_b:
                tied_summary_pairs += 1
                continue


            # -------------------------------------------------
            # Compare source chronology with summary chronology
            # -------------------------------------------------

            source_order = (
                source_time_a < source_time_b
            )

            summary_order = (
                summary_pos_a < summary_pos_b
            )

            if source_order == summary_order:
                correct_pairs += 1
            else:
                reversed_pairs += 1


        evaluated_pairs = (
            correct_pairs + reversed_pairs
        )

        temporal_score = (
            correct_pairs
            / evaluated_pairs
            * 100
            if evaluated_pairs > 0
            else None
        )


        final_temporal_score_records.append(
            {
                "person_id": person_id,
                "workflow": workflow,
                "correct_pairs": correct_pairs,
                "reversed_pairs": reversed_pairs,
                "evaluated_pairs": evaluated_pairs,
                "tied_summary_pairs": tied_summary_pairs,
                "same_source_time_pairs": same_source_time_pairs,
                "missing_event_pairs": missing_event_pairs,
                "temporal_score": temporal_score,
            }
        )


# ---------------------------------------------------------
# Create final dataframe
# ---------------------------------------------------------

final_temporal_scores_df = pd.DataFrame(
    final_temporal_score_records
)


# ---------------------------------------------------------
# Basic validation
# ---------------------------------------------------------

print(
    "Rows:",
    len(final_temporal_scores_df)
)

print(
    "\nPatients per workflow:"
)

print(
    final_temporal_scores_df
    .groupby("workflow")["person_id"]
    .nunique()
)

print(
    "\nMissing temporal scores:",
    final_temporal_scores_df[
        "temporal_score"
    ].isna().sum()
)

print(
    "\nScore range:",
    final_temporal_scores_df[
        "temporal_score"
    ].min(),
    "to",
    final_temporal_scores_df[
        "temporal_score"
    ].max()
)

Rows: 200

Patients per workflow:
workflow
direct              50
hierarchical        50
rag                 50
rag_verification    50
Name: person_id, dtype: int64

Missing temporal scores: 0

Score range: 86.95652173913044 to 100.0


In [39]:
# ---------------------------------------------------------
# Step 19: Temporal consistency descriptive statistics
# ---------------------------------------------------------

final_temporal_summary = (
    final_temporal_scores_df
    .groupby("workflow")["temporal_score"]
    .agg(
        count="count",
        mean="mean",
        median="median",
        std="std",
        min="min",
        max="max"
    )
    .round(2)
)

print(final_temporal_summary)


# ---------------------------------------------------------
# Also inspect total temporal reversals by workflow
# ---------------------------------------------------------

temporal_pair_summary = (
    final_temporal_scores_df
    .groupby("workflow")
    .agg(
        correct_pairs=("correct_pairs", "sum"),
        reversed_pairs=("reversed_pairs", "sum"),
        evaluated_pairs=("evaluated_pairs", "sum"),
        tied_pairs=("tied_summary_pairs", "sum"),
        missing_event_pairs=("missing_event_pairs", "sum")
    )
)

print("\nPair-level totals:")
print(temporal_pair_summary)


# ---------------------------------------------------------
# Number of patients with at least one temporal reversal
# ---------------------------------------------------------

patients_with_reversal = (
    final_temporal_scores_df
    .assign(
        has_reversal=lambda df:
            df["reversed_pairs"] > 0
    )
    .groupby("workflow")["has_reversal"]
    .sum()
)

print("\nPatients with at least one reversed pair:")
print(patients_with_reversal)

                  count   mean  median   std    min    max
workflow                                                  
direct               50  98.98   100.0  1.88  91.89  100.0
hierarchical         50  98.83   100.0  1.99  89.71  100.0
rag                  50  98.91   100.0  2.09  90.62  100.0
rag_verification     50  98.69   100.0  2.77  86.96  100.0

Pair-level totals:
                  correct_pairs  reversed_pairs  evaluated_pairs  tied_pairs  \
workflow                                                                       
direct                     3305              37             3342         862   
hierarchical               3213              40             3253         895   
rag                        2725              28             2753         618   
rag_verification           2574              35             2609         687   

                  missing_event_pairs  
workflow                               
direct                            171  
hierarchical             

In [40]:
# ---------------------------------------------------------
# Step 20: Inspect ceiling effect and paired differences
# ---------------------------------------------------------

from scipy.stats import shapiro
from itertools import combinations

# Put the same 50 patients on rows and workflows on columns.
temporal_pivot = final_temporal_scores_df.pivot(
    index="person_id",
    columns="workflow",
    values="temporal_score"
)

print("Pivot shape:", temporal_pivot.shape)
print(
    "Missing values:",
    temporal_pivot.isna().sum().sum()
)


# ---------------------------------------------------------
# Count perfect (100%) temporal scores
# ---------------------------------------------------------

print("\nPatients with temporal score = 100:")

for workflow in workflows:
    perfect_count = (
        temporal_pivot[workflow] == 100
    ).sum()

    print(
        workflow,
        ":",
        perfect_count,
        "/ 50"
    )


# ---------------------------------------------------------
# Shapiro-Wilk on paired workflow differences
# ---------------------------------------------------------

print("\nShapiro-Wilk tests on paired differences:")

for workflow_a, workflow_b in combinations(
    workflows,
    2
):
    differences = (
        temporal_pivot[workflow_a]
        - temporal_pivot[workflow_b]
    )

    statistic, p_value = shapiro(
        differences
    )

    print(
        f"{workflow_a} - {workflow_b}: "
        f"stat={statistic:.4f}, "
        f"p={p_value:.4g}"
    )

Pivot shape: (50, 4)
Missing values: 0

Patients with temporal score = 100:
direct : 33 / 50
hierarchical : 31 / 50
rag : 34 / 50
rag_verification : 36 / 50

Shapiro-Wilk tests on paired differences:
direct - hierarchical: stat=0.8628, p=3.5e-05
direct - rag: stat=0.8932, p=0.0002892
direct - rag_verification: stat=0.7574, p=1.039e-07
hierarchical - rag: stat=0.8921, p=0.0002658
hierarchical - rag_verification: stat=0.8695, p=5.473e-05
rag - rag_verification: stat=0.6411, p=8.344e-10


In [41]:
# ---------------------------------------------------------
# Step 21: Friedman test for temporal consistency
# ---------------------------------------------------------

from scipy.stats import friedmanchisquare

friedman_statistic, friedman_p = friedmanchisquare(
    temporal_pivot["direct"],
    temporal_pivot["hierarchical"],
    temporal_pivot["rag"],
    temporal_pivot["rag_verification"]
)

print(
    "Friedman statistic:",
    friedman_statistic
)

print(
    "Friedman p-value:",
    friedman_p
)

# Paper-friendly reporting.
if friedman_p < 0.001:
    print(
        "Report as: p < 0.001"
    )
else:
    print(
        f"Report as: p = {friedman_p:.4f}"
    )

Friedman statistic: 0.5632183908045646
Friedman p-value: 0.904798579569909
Report as: p = 0.9048


In [42]:
# ---------------------------------------------------------
# Step 22: Save final temporal evaluation results
# ---------------------------------------------------------

TEMPORAL_SCORES_PATH = (
    EVALUATION_DIR
    / "temporal_patient_scores.csv"
)

TEMPORAL_SUMMARY_PATH = (
    EVALUATION_DIR
    / "temporal_workflow_summary.csv"
)

TEMPORAL_PAIR_SUMMARY_PATH = (
    EVALUATION_DIR
    / "temporal_pair_summary.csv"
)


# Patient-level scores: 50 patients × 4 workflows.
final_temporal_scores_df.to_csv(
    TEMPORAL_SCORES_PATH,
    index=False
)

# Workflow-level descriptive statistics.
final_temporal_summary.to_csv(
    TEMPORAL_SUMMARY_PATH
)

# Pair-level diagnostic totals.
temporal_pair_summary.to_csv(
    TEMPORAL_PAIR_SUMMARY_PATH
)


print("Saved:")
print(TEMPORAL_SCORES_PATH)
print(TEMPORAL_SUMMARY_PATH)
print(TEMPORAL_PAIR_SUMMARY_PATH)

Saved:
/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/temporal_patient_scores.csv
/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/temporal_workflow_summary.csv
/Users/pallavi_chandanshive/projects/clinical-summarization-eval/data/results/evaluation/temporal_pair_summary.csv
